# Data Collection and Processing

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
# Load the downloaded csv from Kaggle
df = pd.read_csv('../raw_data/bot_detection_data.csv')
print(f"Columns:{df.columns.tolist()}")
len(df)

Columns:['User ID', 'Username', 'Tweet', 'Retweet Count', 'Mention Count', 'Follower Count', 'Verified', 'Bot Label', 'Location', 'Created At', 'Hashtags']


50000

### Create the relational database tables

In [15]:
# Table 1: users
users = df[['User ID', 'Username', 'Follower Count', 'Verified', 'Location', 'Bot Label']].drop_duplicates(subset='User ID')
users.columns = ['user_id', 'username', 'follower_count', 'verified', 'location', 'bot_label']
users.to_csv('users.csv', index=False)

# Table 2: tweets
tweets = df[['User ID', 'Tweet', 'Retweet Count', 'Mention Count', 'Created At']].copy()
tweets.insert(0, 'Tweet ID', range(1, len(tweets) + 1))
tweets.columns = ['tweet_id', 'user_id', 'tweet', 'retweet_count', 'mention_count', 'created_at']
tweets.to_csv('tweets.csv', index=False)

# Table 3: hashtags (one row per hashtag per tweet)
tweets_temp = tweets[['tweet_id']].copy()
hashtags_raw = df['Hashtags'].copy()
hashtags_df = pd.DataFrame({
    'tweet_id': tweets_temp['tweet_id'],
    'hashtags': hashtags_raw
})
hashtags_df = hashtags_df.dropna(subset=['hashtags'])
hashtags_df = hashtags_df.assign(hashtag=hashtags_df['hashtags'].str.split()).explode('hashtag')
hashtags_df = hashtags_df[['tweet_id', 'hashtag']].reset_index(drop=True)
hashtags_df.insert(0, 'hashtag_id', range(1, len(hashtags_df) + 1))
hashtags_df.to_csv('hashtags.csv', index=False)

# Table 4: locations (lookup table)
locations = df[['Location']].drop_duplicates().dropna().reset_index(drop=True)
locations.columns = ['location']
locations.insert(0, 'location_id', range(1, len(locations) + 1))

# add location_id back to users table
users = users.merge(locations, on='location', how='left')
users.to_csv('users.csv', index=False)
locations.to_csv('locations.csv', index=False)

In [20]:
# Take a look at the tables
print(users.head())
print(tweets.head())
print(hashtags_df.head())
print(locations.head())
print("\n")
print(len(users))
print(len(tweets))
print(len(hashtags_df))
print(len(locations))

   user_id        username  follower_count  verified      location  bot_label  \
0   132131           flong            2353     False     Adkinston          1   
1   289683  hinesstephanie            9617      True    Sanderston          0   
2   779715      roberttran            4363      True  Harrisonfurt          0   
3   696168          pmason            2242      True  Martinezberg          1   
4   704441          noah87            8438     False  Camachoville          1   

   location_id  
0            1  
1            2  
2            3  
3            4  
4            5  
   tweet_id  user_id                                              tweet  \
0         1   132131  Station activity person against natural majori...   
1         2   289683  Authority research natural life material staff...   
2         3   779715  Manage whose quickly especially foot none to g...   
3         4   696168  Just cover eight opportunity strong policy which.   
4         5   704441                